# Notebook13 fixed-probe evaluation
Inputs: source-bound clean complete48arm12h (or36arm3h) JEPA training. Outputs: artifacts/notebook_13/<run>/stages/learning/stage_evaluation.json, reconstructed predictions, metrics and reports. Official test remains reserved.
Runtime: roughly1h for96probes, included in registered cap. Resume through unchanged notebooks/13_Product_Geometry_JEPA.ipynb launch-product-geometry before original deadline. Imports only ahr-export definitions.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT=Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot={"__file__":str(ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in nbformat.read(ROOT/"fmm_jepa/bootstrap.ipynb",as_version=4).cells:
    if _cell.cell_type=="code" and "ahr-export" in _cell.metadata.get("tags",[]):
        exec(compile(_cell.source,"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)

"""Notebook13 fixed epoch20 probes and six-seed registered primary evidence."""
from pathlib import Path
import copy, hashlib, json, math, sys, time
import numpy as np
import torch

ROOT=Path(__file__).resolve().parents[2]
if 'fmm13_runner' not in sys.modules:
    import types
    _path=ROOT/'experiments/13_product_geometry/runner.ipynb'
    _module=types.ModuleType('fmm13_runner');_module.__file__=str(_path);sys.modules['fmm13_runner']=_module
    for _cell in json.loads(_path.read_text())['cells']:
        if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
            exec(compile(''.join(_cell['source']),str(_path),'exec'),_module.__dict__)
RUNNER=sys.modules['fmm13_runner']
_BASE=RUNNER.load_tagged(ROOT,'experiments/05_scaling/evaluation.ipynb','fmm13_inherited_evaluation')
for _name,_value in list(_BASE.__dict__.items()):
    if not _name.startswith('__') and _name not in globals():globals()[_name]=_value
build_model=RUNNER.build_model


def _evaluation_export_hash(root):
    return _digest([''.join(c['source']) for c in _read(Path(root)/'experiments/13_product_geometry/evaluation.ipynb')['cells']
        if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])])

def _verify_plan_sources(root,plan):
    actual=RUNNER.stage_definition_hashes(root)
    if plan['source_hashes']!=actual:raise ValueError('Notebook13 registered definitions changed')
    if plan.get('consumer_notebook_sha256'):
        code=[''.join(c['source']) for c in _read(Path(root)/plan['resume_notebook'])['cells'] if c['cell_type']=='code']
        if _digest(code)!=plan['consumer_notebook_sha256']:raise ValueError('Registered main notebook code changed')
    return actual

def _stage_jobs(plan,training):
    if training.get('status')!='COMPLETE' or not training.get('equal_updates') or not training.get('paired_traces_within_seed') or not training.get('no_failed_attempts'):
        raise ValueError('Only clean fully paired equal-update training enters evaluation')
    if set(training['arms'])!={a['id'] for a in plan['arms']}:raise ValueError('Training arm identities differ')
    jobs=[]
    for arm in plan['arms']:
        if arm['config']['training']['updates']!=plan['updates'] or arm['config']['probe']['epochs']!=20:raise ValueError('Schedule/probe budget differs')
        if arm['config']['evaluation'].get('heldout_enabled',False):raise ValueError('Official test reserved')
        record=training['arms'][arm['id']]
        if record['successful_updates']!=plan['updates'] or record['attempted_updates']!=plan['updates'] or record['failures']:raise ValueError('Incomplete/failed arm')
        if 'initial' in plan['probe_endpoints']:
            jobs.append({'model_id':arm['id']+'-initial','arm':arm,'updates':0,'control':'initialization','state':record['initial_teacher'],'training_cost':None})
        for milestone in plan['milestones']:
            state=record['milestones'][str(milestone)]
            if state['successful_updates']!=milestone:raise ValueError('Milestone differs')
            jobs.append({'model_id':arm['id']+'-u'+str(milestone),'arm':arm,'updates':milestone,'control':'trained','state':state,'training_cost':state})
    return jobs

def _seed_statistics(rows,value_key='delta_percentage_points',expected_n=None):
    from scipy.stats import t,binomtest
    import itertools
    n=len(rows)
    if n<2 or len({int(row['seed']) for row in rows})!=n or (expected_n is not None and n!=int(expected_n)):raise ValueError('Distinct registered paired training seeds required')
    values=np.asarray([float(row[value_key]) for row in rows],dtype=np.float64)
    if not np.isfinite(values).all():raise ValueError('Nonfinite effect')
    mean=float(values.mean());sd=float(values.std(ddof=1));se=sd/math.sqrt(n);df=n-1
    half=float(t.ppf(.975,df))*se;family_half=float(t.ppf(1-.05/(2*3),df))*se
    statistic=mean/se if se>0 else 0. if mean==0 else None
    p=float(2*t.sf(abs(statistic),df)) if statistic is not None else None
    positive=int((values>0).sum());negative=int((values<0).sum());ties=int((values==0).sum());nonzero=positive+negative
    sign_p=float(binomtest(positive,nonzero,.5,alternative='two-sided').pvalue) if nonzero else 1.
    sums=np.asarray([float(np.dot(signs,values)) for signs in itertools.product((-1.,1.),repeat=n)])
    observed=abs(float(values.sum()));tolerance=1e-12*max(1.,observed,float(np.abs(values).sum()))
    exceed=int((np.abs(sums)>=observed-tolerance).sum())
    return {'n_training_seeds':n,'degrees_of_freedom':df,'mean_percentage_points':mean,'sd_percentage_points':sd,'standard_error_percentage_points':se,
        'approximate_t_ci95_percentage_points':[mean-half,mean+half],
        'bonferroni_family_ci98_333_percentage_points':[mean-family_half,mean+family_half],
        'bonferroni_interval_confidence':1-.05/3,'bonferroni_interval_scope':'Three registered geometry-versus-ViT contrasts; conservative simultaneous family intervals, distinct from Holm p adjustment.',
        't_statistic':statistic,'paired_t_two_sided_p_value':p,'exploratory_two_sided_p_value':p,
        'zero_variance_deltas':se==0,'range_percentage_points':[float(values.min()),float(values.max())],
        'positive_seeds':positive,'negative_seeds':negative,'zero_seeds':ties,
        'exact_two_sided_sign_test':{'p_value':sign_p,'positive_pairs':positive,'negative_pairs':negative,'tied_pairs_excluded':ties,'non_tied_pairs':nonzero,'null':'Equal positive/negative direction probability among non-tied seed pairs; distinct from mean-effect null.'},
        'exact_two_sided_sign_flip_test':{'p_value':exceed/len(sums),'enumerated_assignments':len(sums),'as_or_more_extreme':exceed,'statistic':'Absolute sum of paired seed differences','null_assumption':'Independent paired effects symmetric about zero / valid within-pair exchangeability; conditional robustness analysis, not an assumption-free test.','floating_boundary_tolerance':tolerance},
        'paired_seed_deltas':[{'seed':int(row['seed']),'delta_percentage_points':float(row[value_key])} for row in rows],
        'uncertainty':'Training seeds are replication units; images are shared. Paired t inference assumes appropriate paired-effect distribution. Development images are reused; secondary comparisons exploratory. No guarantee of significance.'}

def analyze_stage_evidence(evaluations,plan=None):
    variants=['vit','learned_kernel','normalized_kernel','product_kernel','sphere_kernel','multi_view_kernel']
    seeds=list(plan['seeds']) if plan is not None else list(range(20261501,20261509))
    if seeds not in (list(range(20261501,20261509)),list(range(20261501,20261507))):raise ValueError('Only registered fresh paired seeds accepted')
    index={(value['variant'],int(value['seed']),int(value['updates'])):value for value in evaluations.values()}
    steps=sorted({key[2] for key in index});final=max(steps,default=0)
    expected_steps=([0,final] if plan is None or 'initial' in plan['probe_endpoints'] else [final])
    candidates=(1000,800,600) if len(seeds)==8 else (600,350,200)
    if final not in candidates or steps!=expected_steps:raise ValueError('Only registered timing-selected initial/final endpoints accepted')
    expected={(variant,seed,step) for variant in variants for seed in seeds for step in steps}
    if set(index)!=expected or len(index)!=len(evaluations):raise ValueError('Every registered unique seed/architecture/endpoint probe required')
    contrasts={};rows=[]
    def paired(name,values):contrasts[name]=_seed_statistics([{'seed':seed,'delta_percentage_points':value} for seed,value in zip(seeds,values)],expected_n=len(seeds))
    if 0 in steps:
        for variant in variants:paired(variant+'_learning_u'+str(final),[_paired_delta(index[(variant,seed,final)],index[(variant,seed,0)]) for seed in seeds])
    comparisons=[(variant,'vit') for variant in variants if variant!='vit']+[
        ('normalized_kernel','learned_kernel'),('product_kernel','normalized_kernel'),('sphere_kernel','product_kernel'),('multi_view_kernel','product_kernel')]
    for candidate,control in comparisons:
        for step in steps:paired(candidate+'_minus_'+control+'_u'+str(step),[_paired_delta(index[(candidate,seed,step)],index[(control,seed,step)]) for seed in seeds])
        if 0 in steps:paired(candidate+'_minus_'+control+'_learning_gain_u'+str(final),[
            _paired_delta(index[(candidate,seed,final)],index[(candidate,seed,0)])-_paired_delta(index[(control,seed,final)],index[(control,seed,0)]) for seed in seeds])
    for variant in variants:
        for seed in seeds:
            initial=_score(index[(variant,seed,0)]) if 0 in steps else None
            for step in steps:
                saved=index[(variant,seed,step)];cost=saved.get('training_cost') or {};perf=cost.get('performance') or {}
                peak=perf.get('peak_cuda_allocated_bytes');diagnostic=saved['representation_diagnostics']['descriptive']
                rows.append({'model_id':saved['model_id'],'variant':variant,'kind':saved['kind'],'seed':seed,'updates':step,
                    'top1_percent':_score(saved),'initial_top1_percent':initial,'gain_percentage_points':_score(saved)-initial if initial is not None else None,
                    'image_exposures':int(cost.get('image_exposures',0)),'cumulative_training_step_seconds':cost.get('cumulative_step_seconds') if step else 0.,
                    'median_training_step_seconds':(perf.get('complete_jepa_step_seconds') or {}).get('median'),'peak_training_GiB':float(peak)/1024**3 if peak is not None else None,
                    'encoder_parameters':saved['parameter_counts']['context'],'trainable_parameters':saved['parameter_counts']['trainable'],
                    'inactive_context_scaffold_parameters':saved['parameter_counts'].get('inactive_context_scaffold',0),'encoder_active_parameters':saved['parameter_counts']['context_active'],
                    'trainable_active_parameters':saved['parameter_counts']['trainable_active'],'feature_std_mean':diagnostic['feature_std_mean'],'effective_rank':diagnostic['effective_rank']})
    sanity={variant:{str(seed):{'finite':bool(index[(variant,seed,final)]['representation_diagnostics']['finite_train_and_development']),
        'not_exactly_constant':bool(index[(variant,seed,final)]['representation_diagnostics']['development_representation_not_exactly_constant'] and index[(variant,seed,final)]['representation_diagnostics']['training_representation_not_exactly_constant'])} for seed in seeds} for variant in variants}
    gate={'passed':all(r['finite'] and r['not_exactly_constant'] for group in sanity.values() for r in group.values()),'status':'NUMERICAL_SANITY_ONLY','representation_checks':sanity,'reason':'No quality-based continuation or model substitution.'}
    family=[variant+'_minus_vit_u'+str(final) for variant in ('product_kernel','sphere_kernel','multi_view_kernel')]
    ordered=sorted(family,key=lambda key:(contrasts[key]['paired_t_two_sided_p_value'] if contrasts[key]['paired_t_two_sided_p_value'] is not None else 1.,key))
    adjusted={};running=0.
    for rank,key in enumerate(ordered):
        raw=contrasts[key]['paired_t_two_sided_p_value'];running=max(running,(len(family)-rank)*(raw if raw is not None else 1.));adjusted[key]=min(1.,running)
    primary_comparisons={}
    for key in family:
        stats=copy.deepcopy(contrasts[key]);stats['holm_adjusted_p_value']=adjusted[key]
        stats['superiority_supported']=bool(stats['paired_t_two_sided_p_value'] is not None and adjusted[key]<.05 and stats['mean_percentage_points']>0 and stats['approximate_t_ci95_percentage_points'][0]>0)
        primary_comparisons[key]=stats
    primary={'registered':True,'family':family,'alpha':.05,'multiplicity':'Holm','n_pairs':len(seeds),'degrees_of_freedom':len(seeds)-1,
        'comparisons':primary_comparisons,'endpoint':'Final fixed epoch20 frozen-EMA development top1 difference in percentage points',
        'primary_mean_test':'Two-sided paired t tests with Holm family correction; positive mean and uncertainty required for superiority.',
        'secondary_robustness_tests':'Exhaustive256 (or64 for registered6pair3hstudy) sign flips and exact sign tests; different assumptions/nulls, not alternate routes to primary success.',
        'scope':'Fresh training seed replications conditional on reused development images; no official test or new dataset claim.'}
    assessment={'status':'REGISTERED_PRIMARY_FAMILY_COMPLETE','registered_pairs':len(seeds),'degrees_of_freedom':len(seeds)-1,
        'primary_analysis':primary,'supported_geometry_variants':[key for key in family if primary_comparisons[key]['superiority_supported']],
        'interpretation':'Report all family contrasts, all seed effects and costs. No automated winner-based extension. Non-significance establishes neither equivalence nor absence of benefit.'}
    return {'contrasts':contrasts,'chart_rows':rows,'learning_gate':gate,'hypothesis_assessment':assessment,'primary_analysis':primary}

def write_stage_figures(artifact,result,summary):
    return {'status':'DEFERRED_TO_BOUND_VISUALIZATION_MODULE','evidence_role':result.get('evidence_role','ACTUAL_EXPERIMENT'),'description':'Source-bound visualization.ipynb writes audited figures after stage completion.'}

def write_stage_report(artifact_dir,result):
    artifact=Path(artifact_dir);final=max(row['updates'] for row in result['chart_rows']);rows=[row for row in result['chart_rows'] if row['updates']==final]
    variants=['vit','learned_kernel','normalized_kernel','product_kernel','sphere_kernel','multi_view_kernel'];summary=[]
    for variant in variants:
        group=[row for row in rows if row['variant']==variant]
        def mean(name):return float(np.mean([row[name] for row in group])) if all(row[name] is not None for row in group) else None
        summary.append({'variant':variant,**{name:mean(source) for name,source in [('top1_percent','top1_percent'),('gain_percentage_points','gain_percentage_points'),('step_seconds','median_training_step_seconds'),('peak_GiB','peak_training_GiB'),('encoder_parameters','encoder_parameters'),('trainable_parameters','trainable_parameters'),('encoder_active_parameters','encoder_active_parameters'),('trainable_active_parameters','trainable_active_parameters'),('feature_std_mean','feature_std_mean'),('effective_rank','effective_rank')]}})
    synthetic=result.get('evidence_role')=='SYNTHETIC_SOFTWARE_VALIDATION'
    title='# SYNTHETIC_SOFTWARE_VALIDATION — invented scores, not research outcomes' if synthetic else '# Notebook13 product geometry JEPA study'
    lines=[title,'',f'All{len(result["evaluations"])} registered probes completed at fixed epoch20. Final successful updates:{final}. Official test: NOT_RUN. Development images reused.','',
        'Three registered geometry-versus-ViT paired training-seed t tests use Holm family correction at alpha0.05. Raw95% t intervals and conservative Bonferroni98.333% family intervals are distinct from Holm adjusted p values. Exact sign-flip and direction tests are secondary.',
        '', '| Variant | Final top1 (%) | Own-init gain (pp) | Step (s) | Peak allocated (GiB) |','|---|---:|---:|---:|---:|']
    for row in summary:
        def show(name):return 'NOT_MEASURED' if row[name] is None else f'{row[name]:.4f}'
        lines.append('|'+row['variant']+'|'+'|'.join(show(name) for name in ('top1_percent','gain_percentage_points','step_seconds','peak_GiB'))+'|')
    lines+=['','| Registered primary comparison | Mean (pp) | Raw95% t CI | Holm p | Supported superiority |','|---|---:|---|---:|---|']
    for key,stats in result['primary_analysis']['comparisons'].items():
        lo,hi=stats['approximate_t_ci95_percentage_points'];lines.append(f'|{key}|{stats["mean_percentage_points"]:+.4f}|[{lo:.4f},{hi:.4f}]|{stats["holm_adjusted_p_value"]:.6g}|{stats["superiority_supported"]}|')
    lines+=['','Training seeds, not image observations, are independent replication units. Original learned and normalized kernels are mechanism controls; other contrasts and initialization gains are exploratory.',
        'Shared initialization covers patch stem, all block MLPs/norms, final norm, predictor, kernel values/output and comparable coordinate maps. Extra geometry parameters are active and reported. Equal exposure does not imply equal compute.',
        'All interactions remain exact dense sums. No FMM acceleration, established intrinsic manifold, image/feature isomorphism or universal superiority claim follows.',
        'In the registered3h final-only design initial model snapshots are retained, but initial accuracy and own-initialization gains are not measured.',
        'Resume through unchanged Notebook13 launch-product-geometry before its original hard deadline; invalid manifests/source identity raise, never silently restart.']
    path=artifact/'stage_results.md';path.write_text('\n'.join(lines)+'\n');atomic_json(artifact/'stage_summary_table.json',summary)
    indexed={row['variant']:row for row in summary};ratios={}
    for variant in variants[1:]:
        ratios[variant+'_over_vit']={name:indexed[variant][name]/indexed['vit'][name] if indexed[variant][name] is not None and indexed['vit'][name] is not None and indexed['vit'][name]>0 else None for name in ('step_seconds','peak_GiB')}
    atomic_json(artifact/'stage_cost_comparisons.json',ratios)
    return {'report':str(path.resolve()),'summary_table':str((artifact/'stage_summary_table.json').resolve()),'cost_comparisons':ratios}

def _evaluate_stage_locked(root, artifact, plan, bundle, training, deadline):
    sources = _verify_plan_sources(root, plan)
    membership = _fixed_membership(root, bundle)
    jobs = _stage_jobs(plan, training)
    stage_role = plan.get("stage_role", plan["stage_id"])
    if stage_role not in STAGE_ROLES: raise ValueError("Unknown stage_role")
    if membership["pretraining_count"] != (65600 if stage_role == "data_volume" else 10100):
        raise ValueError("Pretraining count differs from the registered stage role")
    if int(plan["arms"][0]["config"]["image_size"]) != (256 if stage_role == "resolution" else 128):
        raise ValueError("Stage image resolution differs from declared role")
    identity = {"schema": "notebook13-stage-evaluation/v1", "plan": _canonical(plan),
        "source_hashes": sources, "evaluation_export_hash": _evaluation_export_hash(root),
        "dataset_fingerprint": bundle["dataset"].fingerprint, "membership": membership,
        "jobs": _canonical(jobs), "probe_selection": "fixed_final_epoch20", "official_test": "NOT_RUN"}
    identity_hash = _digest(identity)
    manifest_path = artifact/"stage_evaluation_manifest.json"
    if manifest_path.exists():
        manifest = _read(manifest_path)
        if manifest["identity_hash"] != identity_hash: raise ValueError("Stage evaluation identity changed; refusing stale reuse")
    else:
        manifest = {"identity_hash": identity_hash, "identity": identity, "completed": {}, "started_unix": time.time()}
        atomic_json(manifest_path, manifest)
    evaluations = {}
    for job in jobs:
        model_id, arm = job["model_id"], job["arm"]
        config = copy.deepcopy(arm["config"])
        summary_path = artifact/"evaluation"/(model_id+".json")
        _verified_record(job["state"])
        if model_id in manifest["completed"]:
            evaluations[model_id] = _load_saved_job(summary_path, manifest["completed"][model_id], identity_hash, job, membership)
            continue
        _deadline_check(deadline, 5)
        configure_environment(config)
        atomic_json(artifact/"evaluation_status.json", {"status":"RUNNING", "model_id":model_id,
            "completed_models":len(evaluations), "total_models":len(jobs), "updated_unix":time.time()})
        model = encoder = None
        try:
            _release_semantic(config["device"])
            model = build_model(config, arm["kind"]).cpu()
            parameter_counts={
                'context':sum(p.numel() for p in model.context.parameters()),
                'predictor':sum(p.numel() for p in model.predictor.parameters()),
                'teacher':sum(p.numel() for p in model.target.parameters()),
                'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),
                'inactive_context_scaffold':0}
            parameter_counts['context_active']=parameter_counts['context']-parameter_counts['inactive_context_scaffold']
            parameter_counts['trainable_active']=parameter_counts['trainable']-parameter_counts['inactive_context_scaffold']
            state = torch.load(job["state"]["path"], map_location="cpu", weights_only=job["updates"] == 0)
            if job["updates"] == 0:
                model.target.load_state_dict(state)
            else:
                if int(state["state"]["successful_updates"]) != job["updates"] or state["config"] != arm["config"]:
                    raise ValueError("Checkpoint milestone/config differs from the registered schedule")
                model.load_state_dict(state["model"])
            del state
            encoder = model.target
            model = None
            common = {"dataset_revision":bundle["dataset"].fingerprint, "checkpoint_hash":encoder_state_hash(encoder),
                "encoder_kind":getattr(encoder,"kind",arm["kind"]), "preprocessing":preprocessing_descriptor(config),
                "pooling":"mean_final_layernorm_patches",
                "normalization":"encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
            caches, extraction = {}, []
            for split, ids in (("train", membership["probe_train_ids"]), ("development", membership["development_ids"])):
                _deadline_check(deadline, 2)
                dataset = _DeadlineDataset(EvaluationDataset(bundle["dataset"], ids, config), deadline)
                print(stage_role, model_id, split, len(ids), "fixed labeled feature IDs", flush=True)
                cache = extract_features(encoder, dataset, artifact/"features"/model_id,
                    {**common,"sample_ids":ids}, batch_size=int(config["evaluation"]["batch_size"]), device=config["device"])
                caches[split] = cache
                extraction.append({"split":split, "cache_reused":cache["reused"], **cache["performance"]})
            geometry_ids=list(membership["development_ids"][:2])
            geometry={"status":"NOT_APPLICABLE","reason":"ViT has no declared kernel geometry",
                "sample_ids":geometry_ids,"scope":"Predeclared first two development images; descriptive only, no model/probe selection."}
            if arm["variant"] != "vit":
                geometry_dataset=EvaluationDataset(bundle["dataset"],geometry_ids,config)
                geometry_images=torch.stack([geometry_dataset[i]["image"] for i in range(len(geometry_ids))]).to(next(encoder.parameters()).device)
                backend=RUNNER.load_tagged(root,"experiments/13_product_geometry/encoder.ipynb","fmm13_encoder")
                with torch.no_grad():geometry=backend.kernel_geometry_diagnostics(encoder,geometry_images,max_samples=2)
                geometry.update(sample_ids=geometry_ids,diagnostic_device=str(next(encoder.parameters()).device),scope="Predeclared first two development images; descriptive only, no model/probe selection.")
                del geometry_images,geometry_dataset
            encoder.cpu()
            _release_semantic(config["device"])
            diagnostics = _cached_representation_diagnostics(caches["train"], caches["development"], bundle.get("diagnostic_ids", []))
            result = _fit_fixed_probe(caches["train"], caches["development"],
                root/"checkpoints"/plan["run_id"]/"stages"/plan["stage_id"]/"probes", config, model_id, encoder, deadline, plan)
            saved = {"identity_hash":identity_hash, "model_id":model_id, "arm_id":arm["id"], "kind":arm["kind"], "variant":arm["variant"],
                "seed":int(arm["seed"]), "stage_role":stage_role, "updates":job["updates"], "control":job["control"],
                "source_encoder":job["state"], "feature_caches":{name:_state_record(cache["path"]) for name,cache in caches.items()},
                "probe":result, "parameter_counts":parameter_counts, "representation_diagnostics":diagnostics, "kernel_geometry_diagnostics":geometry, "feature_extraction":extraction,
                "training_cost":job["training_cost"], "heldout_status":"NOT_RUN"}
            write_csv(artifact/"predictions"/(model_id+".csv"), result["predictions"])
            atomic_json(summary_path, saved)
            manifest["completed"][model_id] = _state_record(summary_path)
            atomic_json(manifest_path, manifest)
            evaluations[model_id] = saved
            print(stage_role, model_id, "fixed epoch20 top1", 100*result["summary"]["metrics"]["top1"], flush=True)
        finally:
            model = encoder = None
            _release_semantic(config["device"])
    _deadline_check(deadline)
    if len(evaluations) != len(jobs): raise AssertionError("Every registered endpoint probe is required")
    probe_inits = {x["probe"]["summary"]["initialization_hash"] for x in evaluations.values()}
    probe_budgets = {_digest(x["probe"]["summary"]["budget"]) for x in evaluations.values()}
    probe_orders = {_digest([(r["epoch"],r["split"],r["permutation_hash"]) for r in x["probe"]["metrics"]]) for x in evaluations.values()}
    if len(probe_inits) != 1 or len(probe_budgets) != 1 or len(probe_orders) != 1:
        raise AssertionError("Probe initialization/budget/order is not identical across stage comparisons")
    scientific = analyze_stage_evidence(evaluations,plan)
    result = {"schema":"notebook13-stage-evaluation/v1", "status":"COMPLETE", "run_id":plan["run_id"],
        "stage_id":plan["stage_id"], "stage_role":stage_role, "identity_hash":identity_hash,
        "plan_hash":_digest(plan), "dataset_fingerprint":bundle["dataset"].fingerprint,
        "membership":membership, "evaluations":evaluations, "milestones":sorted({int(value["updates"]) for value in evaluations.values()}),
        "schedule_note":"Registered6or8freshpairedseeds and finalcheckpoint fixed; updates selected only from discardedtiming calibration before quality outcomes. Endpointprobes fixedepoch20, initialprobes onlyin12hdesign.",
        "image_size":int(plan["arms"][0]["config"]["image_size"]), "pretraining_images":len(bundle["train_ids"]),
        "training_design":{"updates":int(plan["updates"]),"effective_batch":plan["arms"][0]["config"]["training"]["microbatch"]*plan["arms"][0]["config"]["training"]["accumulation_steps"],
            "lr":plan["arms"][0]["config"]["training"]["lr"],"ema_start":.99,
            "weight_decay":plan["arms"][0]["config"]["training"]["weight_decay"],
            "warmup_fraction":plan["arms"][0]["config"]["training"]["warmup_fraction"],
            "patch_size":plan["arms"][0]["config"]["model"]["patch_size"],
            "model":copy.deepcopy(plan["arms"][0]["config"]["model"]), "dtype":plan["arms"][0]["config"].get("dtype","float32") },
        "probe_selection":"fixed_final_epoch20", "labels_in_self_supervised_loss":False,
        "official_test":"NOT_RUN", "started_unix":manifest["started_unix"], "completed_unix":time.time(),
        "learning_gate":scientific["learning_gate"], "hypothesis_assessment":scientific["hypothesis_assessment"], "primary_analysis":scientific["primary_analysis"], "contrasts":scientific["contrasts"], "chart_rows":scientific["chart_rows"],
        "report_scope":"Registered fullJEPA sixvariant design, pairedfreshseeds and fixedepoch20 probes. Three geometry-versus-ViT primary family tests withHolm correction; secondary contrasts exploratory; developmentimages reused."}
    atomic_json(artifact/"stage_evaluation.json", result)
    result["presentation"] = write_stage_report(artifact, result)
    atomic_json(artifact/"stage_evaluation.json", result)
    atomic_json(artifact/"evaluation_status.json", {"status":"COMPLETE", "completed_models":len(jobs), "learning_gate":result["learning_gate"], "updated_unix":time.time()})
    return result

def verify_evaluation_smoke(root,artifact,plan,bundle,deadline_unix):
    """Validate real encoder/cache integration and the unchanged fixed20epoch probe path.

    Old07features are used only as validation inputs, never as13quality evidence.
    Fresh11teacher features use two real training and two real development images per architecture.
    """
    root,artifact=Path(root).resolve(),Path(artifact).resolve();artifact.relative_to(root)
    artifact.mkdir(parents=True,exist_ok=True)
    old_root=root/'artifacts/notebook_07/energy-encoder-001'
    old_summary=old_root/'evaluation_recovery_v1/evaluation/s20261011-energy-initial.json'
    old_plan_path=old_root/'stages/learning/plan.json'
    source_summary=_read(old_summary);old_plan=_read(old_plan_path)
    provenance={'summary':_state_record(old_summary),'plan':_state_record(old_plan_path),
        'teacher':source_summary['source_encoder'],'caches':source_summary['feature_caches']}
    identity={'schema':'notebook13-evaluation-smoke/v1','plan_hash':_digest(plan),
        'source_hashes':RUNNER.stage_definition_hashes(root),'validation_only_source07':provenance,
        'fresh_training_pixels':list(bundle['train_ids'][:2]),'fresh_development_pixels':list(bundle['dev_ids'][:2]),
        'scope':'Sixfreshencoderkind realfeature/tinyprobeintegration plus interrupted-resumed fullfinalepoch20probe on old07validationcaches; excludedfrom13scientificresults.'}
    record_path=artifact/'evaluation_smoke.json'
    if record_path.exists():
        record=_read(record_path)
        if record['identity']!=identity or record.get('status')!='PASS':raise ValueError('Evaluation smoke identity/status changed')
        for record_file in record['verified_outputs']:_verified_record(record_file)
        return record
    _deadline_check(deadline_unix,5)
    outputs=[];checks={}
    for variant in plan['variants']:
        arm=next(a for a in plan['arms'] if a['variant']==variant)
        config=copy.deepcopy(arm['config']);configure_environment(config)
        models,initial=RUNNER.initialize_models(config,artifact/'fresh_initial_models'/variant)
        model=models[arm['kind']];del models
        encoder=model.target
        caches={}
        for split,ids in [('train',list(bundle['train_ids'][:2])),('development',list(bundle['dev_ids'][:2]))]:
            dataset=_DeadlineDataset(EvaluationDataset(bundle['dataset'],ids,config),deadline_unix)
            feature_identity={'dataset_revision':bundle['dataset'].fingerprint,'checkpoint_hash':encoder_state_hash(encoder),
                'encoder_kind':encoder.kind,'preprocessing':preprocessing_descriptor(config),
                'pooling':'mean_final_layernorm_patches',
                'normalization':'encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization','sample_ids':ids}
            cache=extract_features(encoder,dataset,artifact/'fresh_initial_features'/variant,feature_identity,batch_size=2,device=config['device'])
            if cache['identity']['encoder_kind']!=variant or cache['features'].shape!=(2,192):raise AssertionError('Fresh encoder integration mismatch')
            caches[split]=cache;outputs.append(_state_record(cache['path']))
        tiny_config=copy.deepcopy(config);tiny_config['probe'].update(epochs=20,batch_size=256)
        tiny_root=root/'checkpoints'/plan['run_id']/'validation'/'tiny_actual_kind_probes'/variant
        with _probe_checkpoint_metadata(plan):
            reference=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'reference',encoder=encoder)
            partial=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'resumed',encoder=encoder,stop_after_updates=1)
            resumed=fit_linear_probe(caches['train'],caches['development'],tiny_root,tiny_config,'resumed',encoder=encoder)
        reference_state=_load_probe_checkpoint(tiny_root/'reference',reference['selected_probe']['identity'])
        resumed_state=_load_probe_checkpoint(tiny_root/'resumed',resumed['selected_probe']['identity'])
        if reference_state['successful_updates']!=20 or resumed_state['successful_updates']!=20 or resumed_state['completed_epochs']!=20:
            raise AssertionError('Tiny real-kind epoch20 probe did not finish')
        for key,tensor in reference_state['classifier'].items():
            if not torch.equal(tensor,resumed_state['classifier'][key]):raise AssertionError('Actual-kind probe resume changed the final classifier')
        x=standardize(caches['development']['features'],resumed_state['standardization'])
        reconstructed=F.linear(x,resumed_state['classifier']['weight'],resumed_state['classifier']['bias'])
        if not torch.isfinite(reconstructed).all():raise AssertionError('Nonfinite tiny real-kind probe reconstruction')
        for name in ('reference','resumed'):outputs.append(_state_record(tiny_root/name/'probe_manifest.json'))
        checks[variant]={'status':'PASS','actual_encoder_kind':encoder.kind,'train_examples':2,'development_examples':2,
            'tiny_probe_completed_epochs':20,'tiny_probe_updates':20,'interrupted_resumed_classifier_bitwise_equal':True,
            'final_logits_reconstructed':True,'quality_result_use':'FORBIDDEN_validation_only'}
        model=encoder=cache=caches=reference=resumed=reference_state=resumed_state=None;_release_semantic(config['device'])
    _deadline_check(deadline_unix,5)
    old_arm=next(a for a in old_plan['arms'] if a['id']==source_summary['arm_id'])
    probe_config=copy.deepcopy(old_arm['config']);configure_environment(probe_config)
    with RUNNER.isolated_rng(int(old_arm['seed'])):
        fixture_runner=RUNNER.load_tagged(root,'experiments/07_energy/runner.ipynb','fmm13_probe_fixture_runner')
        model=fixture_runner.build_model(probe_config,old_arm['kind']).cpu()
    model.target.load_state_dict(torch.load(_verified_record(source_summary['source_encoder']),map_location='cpu',weights_only=True))
    encoder=model.target;caches={}
    for split,record in source_summary['feature_caches'].items():
        path=_verified_record(record);metadata=_read(path.with_suffix('.json'))
        if metadata['sha256']!=digest_file(path):raise ValueError('Validation-only07cache metadata mismatch')
        cache=torch.load(path,map_location='cpu',weights_only=True);validate_feature_cache(cache,metadata['identity'])
        if cache['identity']['encoder_kind']!='energy' or cache['identity']['checkpoint_hash']!=encoder_state_hash(encoder):
            raise ValueError('Validation-only07energy encoder/cache identity mismatch')
        caches[split]=cache
    checkpoint_root=root/'checkpoints'/plan['run_id']/'validation'/'evaluation_smoke_probes'
    model_id='validation-only-07-energy-initial'
    with _probe_checkpoint_metadata(plan):
        partial=fit_linear_probe(caches['train'],caches['development'],checkpoint_root,probe_config,model_id,encoder=encoder,stop_after_updates=17)
    if partial.get('status')!='INTERRUPTED' and not (checkpoint_root/model_id/'probe_manifest.json').exists():
        raise AssertionError('Probe lifecycle failed before resumable checkpoint')
    fixed=_fit_fixed_probe(caches['train'],caches['development'],checkpoint_root,probe_config,model_id,encoder,deadline_unix,plan)
    if fixed['reconciliation']['status']!='PASS' or fixed['summary']['selected_epoch']!=20 or fixed['summary']['successful_probe_updates']!=800:
        raise AssertionError('Fixed epoch20 probe did not reconstruct')
    for metric in ('top1','top5','loss'):
        if not math.isclose(fixed['summary']['metrics'][metric],source_summary['probe']['summary']['metrics'][metric],rel_tol=1e-6,abs_tol=1e-7):
            raise AssertionError('Interrupted/resumed validation probe differs from original fixed-probe result: '+metric)
    outputs.append(fixed['final_probe']);checks['fixed_probe']={'status':'PASS','selected_epoch':20,'successful_updates':800,
        'controlled_interruption_at':17,'predictions_reconstructed':fixed['reconciliation']['predictions_reconstructed'],
        'matched_original07_fixed_probe':True,'quality_result_use':'FORBIDDEN_validation_only'}
    model=encoder=None;_release_semantic(probe_config['device'])
    record={'status':'PASS','identity':identity,'checks':checks,'verified_outputs':outputs,'completed_unix':time.time()}
    atomic_json(record_path,record)
    return record

for _name in ('build_model','_evaluation_export_hash','_verify_plan_sources','_stage_jobs','_seed_statistics',
              'analyze_stage_evidence','write_stage_report','_evaluate_stage_locked'):
    setattr(_BASE,_name,globals()[_name])
_BASE.RESUME_NOTEBOOK=RUNNER.RESUME_NOTEBOOK
_BASE.RESUME_CELL=RUNNER.RESUME_CELL
evaluate_stage=_BASE.evaluate_stage
